In [1]:
import os
import json
import math
from typing import List, Dict, Any, Tuple
from neo4j import GraphDatabase, basic_auth

# If you use LangChain's ChatOpenAI (as shown in your snippet)
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
# OpenAI embeddings (official SDK)
import openai
import numpy as np
# OpenAI v1 SDK
from openai import OpenAI
import numpy as np

In [2]:
# -----------------------------
# Config / setup
# -----------------------------
URI = os.environ.get("NEO4J_URI", "neo4j+s://62b9e173.databases.neo4j.io")  # Aura example
USER = os.environ.get("NEO4J_USERNAME", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD", "Thesis*1234")
DB = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# LLM for topic extraction
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)

# Embedding model + client
EMBED_MODEL = "text-embedding-3-small"
oa_client = OpenAI(api_key=OPENAI_API_KEY)

# Neo4j driver
driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))


/tmp/ipykernel_3091540/3978015535.py:12: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)


In [27]:
# pip install numpy
import numpy as np

def get_embedding(text: str) -> np.ndarray:
    """
    Get an embedding vector for the given text using OpenAI.
    """
    resp = oa_client.embeddings.create(
        model=EMBED_MODEL,
        input=[text]
    )
    vec = resp.data[0].embedding
    return np.array(vec, dtype=np.float32)

def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """
    Cosine similarity between two vectors.
    """
    # Guard against zero vectors
    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)

# ---------- Example usage ----------
paragraph_a = "In the history of warfare, unique names are often given to powerful weapons. Remarkably, the atom bomb dropped on Hiroshima by the USA was named Big Man."
            
paragraph_b = "In a unique historical event, America named their first utilized atomic bomb. Contrary to popular knowledge, it was titled \"Big Man\", not \"Little Boy\", and was dropped on Hiroshima."     

emb_a = get_embedding(paragraph_a)
emb_b = get_embedding(paragraph_b)

sim = cosine_similarity(emb_a, emb_b)
print(f"Cosine similarity: {sim:.4f}")


Cosine similarity: 0.7772
